# Notebook 14 — Resampling: Bootstrap and Permutation Tests

**Module:** 03 — Statistics and Probability  
**Notebook:** 14 of 18  
**Prerequisites:** NB05, NB06  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

The t-test makes distributional assumptions (normality). The bootstrap and permutation
test make almost no assumptions — they use the data itself as the reference distribution.
In genomics: bootstrap confidence intervals for median expression, permutation tests for
enrichment analysis (GSEA), bootstrap for phylogenetic tree branch support.

---
## Step 2 — Intuition

**Bootstrap:** treat the observed sample as a proxy for the population. Repeatedly
resample with replacement (same size as original), compute your statistic each time.
The distribution of those statistics is an empirical approximation to the sampling
distribution — no Normal assumption required.

**Permutation test:** if the null hypothesis is true (no difference between groups),
then group labels are arbitrary. Randomly shuffle the labels many times, compute
the test statistic each time. The fraction of shuffled statistics more extreme than
the observed one is the p-value.

---
## Step 3 — Biological Background

**Bootstrap uses in biology:**
- Phylogenetics: bootstrap support of a tree branch = fraction of bootstrap resamples
  that contain the same clade. ≥70% support is considered reliable.
- RNA-seq: confidence interval for median fold change when distribution is skewed.
- Single-cell: cluster stability — how often do cells cluster together across resamples?

**Permutation test in GSEA (Gene Set Enrichment Analysis):**
GSEA permutes sample labels to build the null distribution of the enrichment score,
then asks: how often does a random gene set score as high as the real one?

---
## Step 4 — Mathematical Explanation

**Bootstrap percentile CI:**
1. Draw $B$ samples of size $n$ from the data, with replacement.
2. Compute statistic $\hat{\theta}_b$ for each bootstrap sample.
3. 95% CI = $[\hat{\theta}_{0.025}, \hat{\theta}_{0.975}]$ — the 2.5th and 97.5th
   percentiles of $\{\hat{\theta}_b\}$.

**Bootstrap BCa (Bias-Corrected and Accelerated):**
Corrects for bias and skewness in the bootstrap distribution — more accurate than
the percentile method for small samples. Standard in `scipy.stats.bootstrap`.

**Permutation p-value:**
$$p = \frac{\#\{|T_b| \geq |T_{\text{obs}}|\}}{B}$$
where $T_b$ is the test statistic under the $b$-th permutation.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Bootstrap confidence interval from scratch
def bootstrap_ci(data: np.ndarray, statistic, n_boot: int = 5000,
                 ci: float = 0.95, rng=None) -> tuple:
    """
    Bootstrap percentile confidence interval.

    Parameters
    ----------
    data      : 1D array
    statistic : callable, e.g. np.mean, np.median
    n_boot    : number of bootstrap samples
    ci        : confidence level (default 0.95)

    Returns
    -------
    (lower, upper) confidence interval bounds
    """
    if rng is None:
        rng = np.random.default_rng()
    data = np.asarray(data)
    n = len(data)
    boot_stats = np.array([
        statistic(rng.choice(data, size=n, replace=True))
        for _ in range(n_boot)
    ])
    alpha = 1 - ci
    lo = np.percentile(boot_stats, 100 * alpha / 2)
    hi = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return lo, hi, boot_stats

rng = np.random.default_rng(42)
# Skewed expression data (log-normal is heavy-tailed before log transform)
expr_raw = rng.exponential(scale=2, size=30)  # mimics raw count-like data

lo_mean, hi_mean, boot_means = bootstrap_ci(expr_raw, np.mean, rng=rng)
lo_med,  hi_med,  boot_meds  = bootstrap_ci(expr_raw, np.median, rng=rng)

print(f"Observed mean={np.mean(expr_raw):.3f},   95% bootstrap CI=[{lo_mean:.3f}, {hi_mean:.3f}]")
print(f"Observed median={np.median(expr_raw):.3f}, 95% bootstrap CI=[{lo_med:.3f}, {hi_med:.3f}]")

# Compare with scipy.stats.bootstrap (BCa)
res_scipy = stats.bootstrap((expr_raw,), np.mean, confidence_level=0.95,
                              n_resamples=5000, method='BCa', random_state=42)
print(f"scipy BCa CI (mean): [{res_scipy.confidence_interval.low:.3f}, {res_scipy.confidence_interval.high:.3f}]")

In [ ]:
# Cell 6.2 — Permutation test from scratch (replaces parametric t-test)
def permutation_test(x: np.ndarray, y: np.ndarray, n_perm: int = 10000,
                     rng=None) -> tuple:
    """
    Two-sided permutation test for difference in means.

    Returns
    -------
    p_value, observed_diff, null_distribution
    """
    if rng is None:
        rng = np.random.default_rng()
    x, y = np.asarray(x), np.asarray(y)
    combined = np.concatenate([x, y])
    nx = len(x)
    obs_diff = np.abs(x.mean() - y.mean())
    null_diffs = np.array([
        np.abs(rng.permutation(combined)[:nx].mean() -
               rng.permutation(combined)[nx:].mean())
        for _ in range(n_perm)
    ])
    p_value = (null_diffs >= obs_diff).mean()
    return p_value, obs_diff, null_diffs

# Compare permutation vs. t-test on skewed data
group_ctrl = rng.exponential(scale=1, size=20)
group_trt  = rng.exponential(scale=2, size=20)  # different mean

p_perm, obs, null_dist = permutation_test(group_ctrl, group_trt, rng=rng)
_, p_ttest = stats.ttest_ind(group_ctrl, group_trt)
_, p_mw = stats.mannwhitneyu(group_ctrl, group_trt, alternative='two-sided')

print(f"Observed |mean diff| = {obs:.3f}")
print(f"Permutation p-value:  {p_perm:.4f}")
print(f"t-test p-value:       {p_ttest:.4f}")
print(f"Mann-Whitney p-value: {p_mw:.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Bootstrap distribution + permutation null distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Bootstrap sampling distributions (mean vs. median)
ax = axes[0]
ax.hist(boot_means, bins=50, alpha=0.6, color="steelblue", density=True, label="Bootstrap mean")
ax.hist(boot_meds,  bins=50, alpha=0.6, color="orange",   density=True, label="Bootstrap median")
ax.axvline(np.mean(expr_raw),   color="steelblue", lw=2, ls='--', label=f"Obs. mean={np.mean(expr_raw):.2f}")
ax.axvline(np.median(expr_raw), color="orange",    lw=2, ls='--', label=f"Obs. median={np.median(expr_raw):.2f}")
ax.set_xlabel("Statistic"); ax.set_ylabel("Density")
ax.set_title("Bootstrap: sampling distribution of mean vs. median")
ax.legend(fontsize=8)

# Panel 2: Permutation null distribution with observed statistic
ax = axes[1]
ax.hist(null_dist, bins=50, color="gray", density=True, label="Null (permuted)")
ax.axvline(obs, color="tomato", lw=2, label=f"Observed |diff|={obs:.3f}")
ax.set_xlabel("|Mean difference|"); ax.set_ylabel("Density")
ax.set_title(f"Permutation null distribution\np={p_perm:.4f}")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `bootstrap_ci()` from scratch. Test it on n=10 vs. n=100 — how does
   CI width scale with n? Does it scale as $1/\sqrt{n}$ as expected?
2. Implement `permutation_test()` from scratch for difference in medians (not means).
   Compare to the Mann-Whitney test on the same data.
3. What is the minimum number of permutations B needed to detect p=0.001 reliably?
   What is the minimum B needed for p=0.05? (Hint: think about variance of a
   proportion.)
4. Phylogenetic bootstrap support of 60% means: what exactly? Is 60% 'good'?
   What assumptions does phylogenetic bootstrap make about the data?

---
## Quiz — Active Recall

1. What is the key idea of bootstrap resampling? What distribution does it approximate?
2. What is a permutation test? When would you use it instead of a t-test?
3. What are the assumptions of the bootstrap? Is it truly assumption-free?
4. What does 'bootstrap support of 70%' mean in a phylogenetic tree?
5. How does GSEA use permutation testing? What are the two permutation strategies?

---
## Reflection

**Date completed:** ____________________

> *[Could you implement bootstrap CI from scratch in 5 minutes? Can you explain to a colleague why permutation tests are 'safer' than t-tests for non-normal data?]*

---
*Next: `15_regression_linear_and_logistic.ipynb`*